In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

PATH = r"D:\Weatherwalay\Day one\weather_hourly_clean.csv"
df = pd.read_csv(PATH, index_col="datetime", parse_dates=True)

# --------------------------------------------------------------
# Build the persistence baseline
# Predict temp_c(t) from temp_c(t-1)
# --------------------------------------------------------------
df["temp_lag_1"] = df["temp_c"].shift(1)

# Drop first row (no lag)
valid = df.dropna(subset=["temp_lag_1", "temp_c"])

# Train/test split by time
split_date = "2016-10-01"
train = valid.loc[:split_date]
test  = valid.loc[split_date:]

print(f"Train rows: {len(train)} ({train.index.min()} → {train.index.max()})")
print(f"Test rows:  {len(test)} ({test.index.min()} → {test.index.max()})")

# Predict: temp(t) = temp(t-1)
baseline_pred = test["temp_lag_1"]
baseline_true = test["temp_c"]

mae  = mean_absolute_error(baseline_true, baseline_pred)
rmse = np.sqrt(mean_squared_error(baseline_true, baseline_pred))

print(f"\n=== Persistence Baseline ===")
print(f"MAE:  {mae:.3f} °C")
print(f"RMSE: {rmse:.3f} °C")

# Compare to "just predict the mean of training"
naive_pred = np.full(len(test), train["temp_c"].mean())
print(f"\n=== Naive Baseline (train mean) ===")
print(f"MAE:  {mean_absolute_error(baseline_true, naive_pred):.3f} °C")
print(f"RMSE: {np.sqrt(mean_squared_error(baseline_true, naive_pred)):.3f} °C")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

PATH = r"D:\Weatherwalay\Day one\weather_hourly_clean.csv"
df = pd.read_csv(PATH, index_col="datetime", parse_dates=True)

# ==================================================================
# 1. Build features
# ==================================================================
feat = pd.DataFrame(index=df.index)

# ---- Target ----
feat["y"] = df["temp_c"]

# ---- Lag features (temperature history) ----
for lag in [1, 2, 3, 6, 12, 24]:
    feat[f"temp_lag_{lag}"] = df["temp_c"].shift(lag)

# ---- Lag features of other variables (at t-1 only) ----
for col in ["dewpt_c", "hum", "pressure_mbar", "wind_kph"]:
    feat[f"{col}_lag_1"] = df[col].shift(1)

# ---- Rolling statistics (only past data) ----
for w in [3, 6, 12, 24]:
    feat[f"temp_roll_mean_{w}"] = df["temp_c"].shift(1).rolling(w).mean()
    feat[f"temp_roll_std_{w}"]  = df["temp_c"].shift(1).rolling(w).std()
    feat[f"temp_roll_min_{w}"]  = df["temp_c"].shift(1).rolling(w).min()
    feat[f"temp_roll_max_{w}"]  = df["temp_c"].shift(1).rolling(w).max()

# ---- Trend: is it warming or cooling? ----
feat["temp_diff_1h"]  = df["temp_c"].shift(1) - df["temp_c"].shift(2)
feat["temp_diff_3h"]  = df["temp_c"].shift(1) - df["temp_c"].shift(4)
feat["temp_diff_24h"] = df["temp_c"].shift(1) - df["temp_c"].shift(25)

# ---- Cyclical time encodings ----
feat["hour_sin"]  = np.sin(2 * np.pi * df["hour_of_day"] / 24)
feat["hour_cos"]  = np.cos(2 * np.pi * df["hour_of_day"] / 24)
feat["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
feat["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# ---- Domain interaction ----
feat["dewpt_spread"] = df["temp_c"].shift(1) - df["dewpt_c"].shift(1)

# ==================================================================
# 2. Drop rows with NaN (first ~25 hours lost to lag/rolling windows)
# ==================================================================
feat = feat.dropna()

print(f"Total rows: {len(feat)}")
print(f"Total features: {feat.shape[1] - 1}")  # -1 for target

# ==================================================================
# 3. Time-based split (NO random shuffle!)
# ==================================================================
split_date = "2016-10-01"
train = feat.loc[:split_date]
test  = feat.loc[split_date:]

X_train = train.drop(columns="y")
y_train = train["y"]
X_test  = test.drop(columns="y")
y_test  = test["y"]

print(f"\nTrain: {len(train)} rows ({train.index.min()} → {train.index.max()})")
print(f"Test:  {len(test)} rows ({test.index.min()} → {test.index.max()})")

# ==================================================================
# 4. Scale features (Ridge needs this)
# ==================================================================
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

# ==================================================================
# 5. Train Ridge regression
# ==================================================================
model = Ridge(alpha=1.0)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)

# ==================================================================
# 6. Evaluate
# ==================================================================
def report(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:<30} MAE: {mae:.3f} °C   RMSE: {rmse:.3f} °C")

print("\n=== Results ===")
report(y_test, test["temp_lag_1"],   "Persistence baseline")
report(y_test, y_pred,               "Ridge regression")

# ==================================================================
# 7. Top feature importances (standardized coefficients)
# ==================================================================
coefs = pd.Series(model.coef_, index=X_train.columns)
coefs_abs = coefs.abs().sort_values(ascending=False)

print("\n=== Top 15 features by |coefficient| ===")
print(coefs_abs.head(15).round(4))

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

PATH = r"D:\Weatherwalay\Day one\weather_hourly_clean.csv"
df = pd.read_csv(PATH, index_col="datetime", parse_dates=True)

# ==================================================================
# Rebuild the same feature matrix (identical to Phase 2)
# ==================================================================
feat = pd.DataFrame(index=df.index)
feat["y"] = df["temp_c"]

for lag in [1, 2, 3, 6, 12, 24]:
    feat[f"temp_lag_{lag}"] = df["temp_c"].shift(lag)

for col in ["dewpt_c", "hum", "pressure_mbar", "wind_kph"]:
    feat[f"{col}_lag_1"] = df[col].shift(1)

for w in [3, 6, 12, 24]:
    feat[f"temp_roll_mean_{w}"] = df["temp_c"].shift(1).rolling(w).mean()
    feat[f"temp_roll_std_{w}"]  = df["temp_c"].shift(1).rolling(w).std()
    feat[f"temp_roll_min_{w}"]  = df["temp_c"].shift(1).rolling(w).min()
    feat[f"temp_roll_max_{w}"]  = df["temp_c"].shift(1).rolling(w).max()

feat["temp_diff_1h"]  = df["temp_c"].shift(1) - df["temp_c"].shift(2)
feat["temp_diff_3h"]  = df["temp_c"].shift(1) - df["temp_c"].shift(4)
feat["temp_diff_24h"] = df["temp_c"].shift(1) - df["temp_c"].shift(25)

feat["hour_sin"]  = np.sin(2 * np.pi * df["hour_of_day"] / 24)
feat["hour_cos"]  = np.cos(2 * np.pi * df["hour_of_day"] / 24)
feat["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
feat["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

feat["dewpt_spread"] = df["temp_c"].shift(1) - df["dewpt_c"].shift(1)

feat = feat.dropna()

# ==================================================================
# Split
# ==================================================================
split_date = "2016-10-01"
train = feat.loc[:split_date]
test  = feat.loc[split_date:]

X_train = train.drop(columns="y")
y_train = train["y"]
X_test  = test.drop(columns="y")
y_test  = test["y"]

# ==================================================================
# Train LightGBM
# ==================================================================
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# ==================================================================
# Evaluate
# ==================================================================
def report(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:<30} MAE: {mae:.3f} °C   RMSE: {rmse:.3f} °C")

print("=== Results ===")
report("Persistence baseline",  y_test, test["temp_lag_1"])
report("Ridge regression",      y_test, y_pred)  # (this is LGBM)

# ==================================================================
# Feature importance (gain — most useful)
# ==================================================================
imp = pd.Series(model.booster_.feature_importance(importance_type="gain"),
                index=X_train.columns)
print("\n=== Top 15 features by gain ===")
print(imp.sort_values(ascending=False).head(15).round(1))

In [ ]:
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

# ==================================================================
# Rebuild the feature matrix (same as before)
# ==================================================================
PATH = r"D:\Weatherwalay\Day one\weather_hourly_clean.csv"
df = pd.read_csv(PATH, index_col="datetime", parse_dates=True)

feat = pd.DataFrame(index=df.index)
feat["y"] = df["temp_c"]

for lag in [1, 2, 3, 6, 12, 24]:
    feat[f"temp_lag_{lag}"] = df["temp_c"].shift(lag)

for col in ["dewpt_c", "hum", "pressure_mbar", "wind_kph"]:
    feat[f"{col}_lag_1"] = df[col].shift(1)

for w in [3, 6, 12, 24]:
    feat[f"temp_roll_mean_{w}"] = df["temp_c"].shift(1).rolling(w).mean()
    feat[f"temp_roll_std_{w}"]  = df["temp_c"].shift(1).rolling(w).std()
    feat[f"temp_roll_min_{w}"]  = df["temp_c"].shift(1).rolling(w).min()
    feat[f"temp_roll_max_{w}"]  = df["temp_c"].shift(1).rolling(w).max()

feat["temp_diff_1h"]  = df["temp_c"].shift(1) - df["temp_c"].shift(2)
feat["temp_diff_3h"]  = df["temp_c"].shift(1) - df["temp_c"].shift(4)
feat["temp_diff_24h"] = df["temp_c"].shift(1) - df["temp_c"].shift(25)

feat["hour_sin"]  = np.sin(2 * np.pi * df["hour_of_day"] / 24)
feat["hour_cos"]  = np.cos(2 * np.pi * df["hour_of_day"] / 24)
feat["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
feat["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

feat["dewpt_spread"] = df["temp_c"].shift(1) - df["dewpt_c"].shift(1)

feat = feat.dropna()

# ==================================================================
# Three-way split:
#   Train  →  ends 2016-06-30    (6 months)
#   Val    →  2016-07-01 to 09-30 (3 months)
#   Test   →  2016-10-01 to 12-31 (3 months)
# ==================================================================
split_train_val = "2016-07-01"
split_val_test  = "2016-10-01"

train = feat.loc[:split_train_val]
val   = feat.loc[split_train_val:split_val_test].iloc[1:]   # avoid overlap
test  = feat.loc[split_val_test:].iloc[1:]

X_tr, y_tr = train.drop(columns="y"), train["y"]
X_vl, y_vl = val.drop(columns="y"),   val["y"]
X_te, y_te = test.drop(columns="y"),  test["y"]

print(f"Train: {len(X_tr)} rows  ({train.index.min()} → {train.index.max()})")
print(f"Val:   {len(X_vl)} rows  ({val.index.min()} → {val.index.max()})")
print(f"Test:  {len(X_te)} rows  ({test.index.min()} → {test.index.max()})")

# ==================================================================
# Regularized LightGBM with early stopping on the validation set
# ==================================================================
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=15,             # was 31
    max_depth=5,               # cap depth
    min_child_samples=50,      # was 20
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,      # was 0.8
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=42,
    verbose=-1,
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_vl, y_vl)],
    eval_metric="mae",
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

print(f"\nBest iteration: {model.best_iteration_}")

# ==================================================================
# Evaluate on all three splits
# ==================================================================
def report(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:<25} MAE: {mae:.3f}  RMSE: {rmse:.3f}")

print("\n=== Results ===")
report("Persistence",      y_te, test["temp_lag_1"])
report("Ridge (baseline)", y_te, y_te * 0 + 0)  # skip; you already have this
report("LGBM – Train",     y_tr, model.predict(X_tr))
report("LGBM – Val",       y_vl, model.predict(X_vl))
report("LGBM – Test",      y_te, model.predict(X_te))

gap = mean_absolute_error(y_te, model.predict(X_te)) - mean_absolute_error(y_tr, model.predict(X_tr))
print(f"\nTrain→Test gap: {gap:.3f}")

In [ ]:
# fair ridge vs light GBM
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ==================================================================
# 1. REBUILD FEATURES (identical to before)
# ==================================================================
PATH = r"D:\Weatherwalay\Day one\weather_hourly_clean.csv"
df = pd.read_csv(PATH, index_col="datetime", parse_dates=True)

feat = pd.DataFrame(index=df.index)
feat["y"] = df["temp_c"]

for lag in [1, 2, 3, 6, 12, 24]:
    feat[f"temp_lag_{lag}"] = df["temp_c"].shift(lag)

for col in ["dewpt_c", "hum", "pressure_mbar", "wind_kph"]:
    feat[f"{col}_lag_1"] = df[col].shift(1)

for w in [3, 6, 12, 24]:
    feat[f"temp_roll_mean_{w}"] = df["temp_c"].shift(1).rolling(w).mean()
    feat[f"temp_roll_std_{w}"]  = df["temp_c"].shift(1).rolling(w).std()
    feat[f"temp_roll_min_{w}"]  = df["temp_c"].shift(1).rolling(w).min()
    feat[f"temp_roll_max_{w}"]  = df["temp_c"].shift(1).rolling(w).max()

feat["temp_diff_1h"]  = df["temp_c"].shift(1) - df["temp_c"].shift(2)
feat["temp_diff_3h"]  = df["temp_c"].shift(1) - df["temp_c"].shift(4)
feat["temp_diff_24h"] = df["temp_c"].shift(1) - df["temp_c"].shift(25)

feat["hour_sin"]  = np.sin(2 * np.pi * df["hour_of_day"] / 24)
feat["hour_cos"]  = np.cos(2 * np.pi * df["hour_of_day"] / 24)
feat["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
feat["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

feat["dewpt_spread"] = df["temp_c"].shift(1) - df["dewpt_c"].shift(1)

feat = feat.dropna()

# ==================================================================
# 2. SAME 3-WAY SPLIT AS LAST TIME
# ==================================================================
split_train_val = "2016-07-01"
split_val_test  = "2016-10-01"

train = feat.loc[:split_train_val]
val   = feat.loc[split_train_val:split_val_test].iloc[1:]
test  = feat.loc[split_val_test:].iloc[1:]

X_tr, y_tr = train.drop(columns="y"), train["y"]
X_vl, y_vl = val.drop(columns="y"),   val["y"]
X_te, y_te = test.drop(columns="y"),  test["y"]

# ==================================================================
# 3. TRAIN BOTH MODELS ON THE SAME SPLIT
# ==================================================================
# --- Ridge ---
scaler = StandardScaler().fit(X_tr)
ridge = Ridge(alpha=1.0).fit(scaler.transform(X_tr), y_tr)
ridge_pred_te = ridge.predict(scaler.transform(X_te))
ridge_pred_vl = ridge.predict(scaler.transform(X_vl))
ridge_pred_tr = ridge.predict(scaler.transform(X_tr))

# --- LightGBM (same regularized settings as last run) ---
lgbm = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.02,
    num_leaves=15, max_depth=5, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=0.5, random_state=42, verbose=-1,
)
lgbm.fit(
    X_tr, y_tr,
    eval_set=[(X_vl, y_vl)],
    eval_metric="mae",
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
lgbm_pred_te = lgbm.predict(X_te)
lgbm_pred_vl = lgbm.predict(X_vl)
lgbm_pred_tr = lgbm.predict(X_tr)

# ==================================================================
# 4. HEAD-TO-HEAD COMPARISON
# ==================================================================
def report(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:<28} MAE: {mae:.3f}   RMSE: {rmse:.3f}")

print("=" * 60)
print("MODEL COMPARISON — same 3-way split")
print("=" * 60)
print("\nPersistence:")
report("  Test", y_te, test["temp_lag_1"])

print("\nRidge:")
report("  Train", y_tr, ridge_pred_tr)
report("  Val",   y_vl, ridge_pred_vl)
report("  Test",  y_te, ridge_pred_te)

print(f"\nLightGBM (best iter = {lgbm.best_iteration_}):")
report("  Train", y_tr, lgbm_pred_tr)
report("  Val",   y_vl, lgbm_pred_vl)
report("  Test",  y_te, lgbm_pred_te)

# ==================================================================
# 5. ERROR ANALYSIS — use the best model on test
# ==================================================================
best_name = "Ridge" if mean_absolute_error(y_te, ridge_pred_te) < \
                       mean_absolute_error(y_te, lgbm_pred_te) else "LightGBM"
best_pred = ridge_pred_te if best_name == "Ridge" else lgbm_pred_te

print(f"\nUsing {best_name} for error analysis.")

residuals = y_te - best_pred            # signed errors
abs_err   = residuals.abs()

# Rate of change during the target hour
change = (df["temp_c"] - df["temp_c"].shift(1)).reindex(y_te.index)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# ---- 1. Predicted vs Actual ----
ax = axes[0, 0]
ax.scatter(y_te, best_pred, alpha=0.3, s=8, color="steelblue")
lims = [min(y_te.min(), best_pred.min()) - 2,
        max(y_te.max(), best_pred.max()) + 2]
ax.plot(lims, lims, "r--", lw=1)
ax.set_xlabel("actual temp_c"); ax.set_ylabel("predicted")
ax.set_title(f"{best_name} — Predicted vs Actual (test)")

# ---- 2. Residual distribution ----
ax = axes[0, 1]
ax.hist(residuals, bins=60, edgecolor="black", alpha=0.8)
ax.axvline(0, color="red", lw=0.8)
ax.set_title(f"Residuals — mean={residuals.mean():.3f}, "
             f"std={residuals.std():.3f}, MAE={abs_err.mean():.3f}")
ax.set_xlabel("actual − predicted")

# ---- 3. Residuals over time ----
ax = axes[1, 0]
ax.plot(y_te.index, residuals, lw=0.4, color="darkred")
ax.axhline(0, color="black", lw=0.5)
ax.set_title("Residuals over time (test period)")
ax.set_ylabel("residual")

# ---- 4. Error vs rate of change ----
ax = axes[1, 1]
ax.scatter(change.abs(), abs_err, alpha=0.3, s=8, color="darkgreen")
ax.set_xlabel("|Δtemp| in target hour")
ax.set_ylabel("|residual|")
ax.set_title("Error vs rate of change")

plt.tight_layout()
plt.show()

# ---- 5. Quantitative breakdown by rate of change ----
print("\n=== Error by rate of change ===")
bins = [(0, 0.5), (0.5, 1.0), (1.0, 2.0), (2.0, 4.0), (4.0, 100)]
for lo, hi in bins:
    mask = (change.abs() >= lo) & (change.abs() < hi)
    if mask.sum() > 0:
        print(f"|Δtemp| in [{lo:>3.1f}, {hi:>4.1f}) °C: "
              f"n={mask.sum():>4}   MAE={abs_err[mask].mean():.3f}")

# ---- 6. Worst 10 hours ----
print("\n=== 10 worst predictions ===")
worst = pd.DataFrame({
    "actual": y_te,
    "predicted": best_pred,
    "residual": residuals,
    "delta_temp": change,
}).reindex(abs_err.sort_values(ascending=False).head(10).index)
print(worst.round(2))